## Two circles experiments
Adapted from

https://scikit-learn.org/stable/auto_examples/decomposition/plot_kernel_pca.html


In [1]:
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from src.kernel_pca import KernelPCA, RFF_KernelPCA

import numpy as np
import matplotlib.pyplot as plt
import src.utils.plotting_params  # Load the plotting parameters

# automatic reload
%load_ext autoreload
%autoreload 2

In [2]:
# output folder
out_dir = "../../writeup/Kernel_PCA_Preimage/figures/"

## Generate training and test data

This is a standard example where PCA fails to linearly separate the data

In [3]:
X, y = make_circles(n_samples=1_000, factor=0.3, noise=0.05, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=0)

**Visualize the data**

In [4]:
fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(4., 2.), sharex=True, sharey=True, constrained_layout=True)

ax1.scatter(X_train[:, 0], X_train[:, 1], c=y_train, s=5)
ax1.set_xlabel(r"$x_1$")
ax1.set_ylabel(r"$x_2$")
ax1.set_aspect('equal')
ax1.set_title("Training data", loc="left", fontsize=12)

ax2.scatter(X_test[:, 0], X_test[:, 1], c=y_test, s=5)
ax2.set_xlabel(r"$x_1$")
ax2.set_aspect('equal')
ax2.set_title("Test data", loc="left", fontsize=12)

ax1.grid(False)
ax2.grid(False)
fig.savefig(out_dir + "two_circles_train_test_data.pdf", format="pdf", bbox_inches="tight")
plt.close(fig)

## Feature learning: PCA `vs` Kernel-PCA `vs` Kernel-PCA-RFF 

In [28]:
pca = PCA(n_components=2)
kernel_pca = KernelPCA(n_components=None, kernel="rbf", gamma=10)
rff_kpca = RFF_KernelPCA(n_components=kernel_pca.n_components, n_rffs=200, gamma=kernel_pca.gamma, gamma_inv=None, random_state=42)


X_test_pca = pca.fit(X_train).transform(X_test)
X_test_kpca = kernel_pca.fit(X_train).transform(X_test)
rff_kpca.fit(X_train)
X_test_rff_kpca = rff_kpca.transform(X_test)

In [29]:
methods = ["Original test data", "PCA projection", "KPCA projection", "KPCA-RFF projection"]
embeddings = [X_test, X_test_pca, X_test_kpca, X_test_rff_kpca]
x_labels = [r'$x_1$', 'PC 1', 'PC 1', 'PC 1']
y_labels = [r'$x_2$', 'PC 2', '', '']

fig = plt.figure(figsize=(8, 2.0), constrained_layout=True)
gs = fig.add_gridspec(
    1, 4,
    wspace=0.1,
    top=0.82,                                 # fixed top margin for titles
    bottom=0.18,
)

axes = [fig.add_subplot(gs[i]) for i in range(4)]

for ax, embedding, x_label, y_label in zip(axes, embeddings, x_labels, y_labels):
    ax.scatter(
        embedding[:, 0], embedding[:, 1],
        c=y_test,
        s=15, alpha=0.8,
        linewidths=0.3, edgecolors="white",
    )
    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    #ax.spines[["top", "right", "left", "bottom"]].set_visible(False)
    ax.grid(False)

for ax, label, title in zip(axes, ["(a)", "(b)", "(c)", "(d)"], methods):
    ax.set_title(f"{label} {title}", loc="left", fontsize=9, linespacing=1.4)
    
fig.savefig(out_dir + "two_circles_projection.pdf", format="pdf", bbox_inches="tight")
plt.close(fig)

## Reconstruction

In [30]:
alpha = 1e-6  # ridge parameter
X_reconstructed_pca = pca.inverse_transform(pca.transform(X_test))
X_reconstructed_kpca = kernel_pca.inverse_transform_bws(kernel_pca.transform(X_test), alpha=alpha)

kernel_pca.fit_inverse_transform_ours(alpha=0.0)
X_reconstructed_kpca_new = kernel_pca.inverse_transform_ours(kernel_pca.transform(X_test))
X_reconstructed_rff_ours = rff_kpca.inverse_transform_ours(X_test_rff_kpca, alpha=0.0)
X_reconstructed_rff_bws = rff_kpca.inverse_transform_bws(X_test_rff_kpca, alpha=alpha)
X_reconstructed_rff_tr = rff_kpca.inverse_transform_tr(X_test_rff_kpca, alpha=0.0)

In [31]:
# calculate MSE 
mse_kpca = np.mean((X_test - X_reconstructed_kpca) ** 2)
mse_kpca_new = np.mean((X_test - X_reconstructed_kpca_new) ** 2)
mse_rff_ours = np.mean((X_test - X_reconstructed_rff_ours) ** 2)
mse_rff_bws = np.mean((X_test - X_reconstructed_rff_bws) ** 2)
mse_rff_tr = np.mean((X_test - X_reconstructed_rff_tr) ** 2)

In [33]:
titles = [
    (r"Original test data", ""),
    (rf"KPCA (B.W.S.)", rf"MSE: {mse_kpca:.1e}"),
    (rf"KPCA (Ours)",  rf"MSE: {mse_kpca_new:.1e}"),
    (rf"KPCA-RFF (B.W.S.)", rf"MSE: {mse_rff_bws:.1e}"),
    (rf"KPCA-RFF (T.R.)", rf"MSE: {mse_rff_tr:.1e}"),
    (rf"KPCA-RFF (Ours)", rf"MSE: {mse_rff_ours:.1e}"),
]
data_all = [X_test, X_reconstructed_kpca, X_reconstructed_kpca_new, X_reconstructed_rff_bws, X_reconstructed_rff_tr, X_reconstructed_rff_ours]

fig, axes = plt.subplots(ncols=len(data_all), sharex=True, sharey=True, figsize=(1.35*len(data_all), 2.0), constrained_layout=True)

for ax, data, (title, mse_label) in zip(
    axes, data_all,  titles
):
    ax.scatter(
        data[:, 0], data[:, 1],
        c=y_test,
        s=15, alpha=0.8,
        linewidths=0.3, edgecolors="white",
    )
    ax.set_aspect("equal")
    ax.grid(False)

    # title
    ax.text(
        0.0, 1.08, title,
        transform=ax.transAxes,
        fontsize=10, fontweight="normal",
        va="bottom", ha="left",
    )

    # MSE as legend
    if mse_label:
        from matplotlib.lines import Line2D
        ax.legend(
            handles=[Line2D([], [], color="none")],
            labels=[mse_label],
            loc="upper right",
            bbox_to_anchor=(1.0, 1.12),   # x=1.0 keeps it right-aligned, y=1.15 pushes it up
            fontsize=8.5,
            frameon=True,
            framealpha=0.3,
            edgecolor="none",
            handlelength=0,
            handletextpad=0,
        )
    fig.supxlabel(r'$x_1$', fontsize=9)
    fig.supylabel(r'$x_2$', fontsize=9)
fig.savefig(out_dir + "two_circles_reconstruction.pdf", format="pdf", bbox_inches="tight")
plt.close(fig)